# RAG + LLM Source Mapping Pipeline

This pipeline combines:
1. **RAG Source Finding**: Retrieve top 10 candidate sources using embedding similarity
2. **LLM Source Mapping**: Generate source properties for each candidate using `SourcePropertyFind_mapWithDescription`
3. **LLM Selection**: Select best 3 sources based on generated mappings


## Setup and Imports


In [1]:
import sys
import os
import pandas as pd
import numpy as np
import json
import re
import ast
from dotenv import load_dotenv
import dspy
from tqdm.auto import tqdm

# Load environment variables
load_dotenv()

# Add paths for imports
sys.path.append('../../stage1_analysis/source_finding')
sys.path.append('../../stage1_analysis/mapping_generation')

from rag_source_finder import RAGSourceFinder
from easy_llm_importer import create_client, DSPyAdapter, LLMClient

# Data path
DATA_PATH = "../../data/SCAR_cleaned_manually.csv"

print("Imports loaded successfully!")


Imports loaded successfully!


c:\Users\maria\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configure LLM client
client = create_client()
llm_client = LLMClient()

# Configure DSPy with Llama 405B
MODEL_NAME = "llama-3.1-405b-instruct"
adapter = DSPyAdapter(client, model_name=MODEL_NAME)
lm = adapter.get_dspy_lm()
dspy.settings.configure(lm=lm)

print(f"DSPy configured with {MODEL_NAME}")


DSPy configured with llama-3.1-405b-instruct


## DSPy Signature for Source Property Mapping


In [3]:
# Source Property Mapping Signature (from Mapping_analysis_v3.ipynb)
class SourcePropertyFind_mapWithDescription(dspy.Signature):
    """Given an unfamiliar concept with its properties and descriptions, find corresponding properties from a familiar concept."""
    
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    description_of_unfamiliar_concept: str = dspy.InputField(desc="Detailed description or context about the unfamiliar concept")
    properties_of_unfamiliar_concept: list[str] = dspy.InputField(desc="List of key properties that characterize the unfamiliar concept. Each property is 1-2 words maximum.")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    description_of_familiar_concept: str = dspy.InputField(desc="Detailed description or context about the familiar concept")
    mapped_source_properties: dict[str, str] = dspy.OutputField(desc="Dictionary mapping each unfamiliar concept property to corresponding familiar concept property (1-2 words each)")

print("DSPy signature defined!")


DSPy signature defined!


## Initialize RAG Source Finder


In [4]:
# Initialize RAG source finder
finder = RAGSourceFinder(embedding_mode="name_properties")
finder.load_corpus_from_csv(DATA_PATH)
finder.embed_corpus()

print(f"RAG finder ready with {len(finder.corpus_df)} sources")


Loaded 333 unique sources from corpus (mode: name_properties)
Generating embeddings for corpus...
Generated 333 embeddings
RAG finder ready with 333 sources


In [5]:
# Load SCAR dataset
df_scar = pd.read_csv(DATA_PATH)

# Build source description lookup
source_desc_lookup = {}
for _, row in df_scar.iterrows():
    source_name = row['system_b']
    if source_name not in source_desc_lookup:
        source_desc_lookup[source_name] = row['system_b_background'] if pd.notna(row['system_b_background']) else ""

# Helper to extract target properties from mappings_parsed
def get_target_properties(row):
    if pd.notna(row.get('mappings_parsed')):
        try:
            mappings = ast.literal_eval(row['mappings_parsed'])
            props = [pair[0] for pair in mappings if len(pair) > 0]
            return props
        except:
            pass
    return []

print(f"Loaded {len(df_scar)} examples from SCAR dataset")
print(f"Source descriptions available: {len(source_desc_lookup)}")


Loaded 400 examples from SCAR dataset
Source descriptions available: 333


## Pipeline Functions


In [6]:
def generate_source_mappings(target_name, target_desc, target_properties, source_name, source_desc):
    """
    Use LLM to generate source properties that map to target properties.
    Returns dict mapping target_prop -> generated_source_prop
    """
    try:
        predictor = dspy.ChainOfThought(SourcePropertyFind_mapWithDescription)
        result = predictor(
            unfamiliar_concept=target_name,
            description_of_unfamiliar_concept=target_desc,
            properties_of_unfamiliar_concept=target_properties,
            familiar_concept=source_name,
            description_of_familiar_concept=source_desc
        )
        return result.mapped_source_properties
    except Exception as e:
        print(f"Mapping failed for {source_name}: {e}")
        return {}


def llm_select_best_sources(target_name, target_properties, target_desc, candidates_with_mappings, top_k=3):
    """
    Use LLM to select best 3 sources from candidates.
    candidates_with_mappings: list of (source_name, generated_properties_dict)
    """
    # Format candidates with their generated properties
    candidates_text = "\n".join([
        f"{i+1}. {name} - Generated Properties: {', '.join(props.values()) if props else 'None'}"
        for i, (name, props) in enumerate(candidates_with_mappings)
    ])
    
    prompt = f"""You are selecting analogous sources for a scientific analogy.

Target concept: {target_name}
Target properties: {', '.join(target_properties)}

Candidate source concepts with their generated analogous properties:
{candidates_text}

Select the {top_k} BEST candidates that would make good analogies for the target concept.
Return ONLY a JSON array of the candidate numbers (1-indexed), e.g., [1, 3, 5]
Do not include any explanation, just the JSON array.
"""
    
    try:
        response = llm_client.chat(
            model_name=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=100
        )
        response_clean = response.strip()
        
        # Try to parse JSON
        try:
            selected = json.loads(response_clean)
        except json.JSONDecodeError:
            # Try to find array pattern
            match = re.search(r'\[[\d,\s]+\]', response_clean)
            if match:
                selected = json.loads(match.group())
            else:
                # Extract numbers
                numbers = re.findall(r'\b([1-9]|10)\b', response_clean)
                selected = [int(n) for n in numbers[:top_k]]
        
        if not selected:
            return list(range(1, top_k + 1))  # Fallback to first 3
        
        return [i for i in selected if 1 <= i <= len(candidates_with_mappings)][:top_k]
    except Exception as e:
        print(f"LLM selection failed: {e}")
        return list(range(1, top_k + 1))


def run_pipeline(row):
    """
    Run full pipeline on a single row:
    1. RAG retrieval (top 10)
    2. Generate source mappings for each candidate
    3. LLM selects best 3
    """
    target_name = row['system_a']
    target_desc = row['system_a_background']
    target_properties = get_target_properties(row)
    gold_source = row['system_b']
    
    # Step 1: RAG retrieval
    candidates = finder.find_source(
        target_name=target_name,
        target_background=target_desc,
        target_properties=", ".join(target_properties),
        top_k=10
    )
    
    # Step 2: Generate source mappings for each candidate
    candidates_with_mappings = []
    for candidate in candidates:
        source_desc = source_desc_lookup.get(candidate.name, "")
        mappings = generate_source_mappings(
            target_name, target_desc, target_properties,
            candidate.name, source_desc
        )
        candidates_with_mappings.append((candidate.name, mappings))
    
    # Step 3: LLM selects best 3
    selected_indices = llm_select_best_sources(
        target_name, target_properties, target_desc,
        candidates_with_mappings, top_k=3
    )
    
    # Get selected sources
    selected_sources = [candidates_with_mappings[i-1] for i in selected_indices if i <= len(candidates_with_mappings)]
    selected_names = [name for name, _ in selected_sources]
    
    # Find gold rank in selected
    gold_rank = -1
    for i, name in enumerate(selected_names):
        if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
            gold_rank = i + 1
            break
    
    return {
        'id': row['id'],
        'target': target_name,
        'target_properties': target_properties,
        'gold_source': gold_source,
        'top_10_sources': [c.name for c in candidates],
        'top_10_scores': [c.similarity_score for c in candidates],
        'candidates_with_mappings': candidates_with_mappings,
        'selected_indices': selected_indices,
        'selected_sources': selected_names,
        'selected_mappings': [props for _, props in selected_sources],
        'gold_rank': gold_rank
    }

print("Pipeline functions defined!")


Pipeline functions defined!


## Run Pipeline on SCAR Dataset


In [7]:
# Run pipeline on full dataset
results = []

print(f"Running pipeline on {len(df_scar)} examples...")
for idx, row in tqdm(df_scar.iterrows(), total=len(df_scar), desc="Processing"):
    result = run_pipeline(row)
    results.append(result)
    
    # Progress update every 50
    if (idx + 1) % 50 == 0:
        found_so_far = sum(1 for r in results if r['gold_rank'] > 0)
        print(f"Progress: {idx+1}/{len(df_scar)} - Hit rate so far: {100*found_so_far/len(results):.1f}%")

print(f"\nPipeline complete! Processed {len(results)} examples.")


Running pipeline on 400 examples...


Processing:   0%|          | 0/400 [00:00<?, ?it/s]

Processing:  12%|█▎        | 50/400 [1:09:07<8:43:15, 89.70s/it]

Progress: 50/400 - Hit rate so far: 42.0%


Processing:  20%|██        | 81/400 [1:51:40<6:44:00, 75.99s/it]2025/12/17 19:15:10 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.


...

LLM selection failed: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 100 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_33QcGvgPvn2aVps1j9TdBlgP5Ii'}
Mapping failed for Reactor: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_tokens. You requested up to 4000 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits
Mapping failed for atom: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_tokens. You requested up to 4000 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits
Mapping failed for planet: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fe

Processing: 100%|█████████▉| 399/400 [11:02:36<01:42, 102.32s/it]

LLM selection failed: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 100 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_33QcGvgPvn2aVps1j9TdBlgP5Ii'}
Mapping failed for Reactor: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_tokens. You requested up to 4000 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits
Mapping failed for atom: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_tokens. You requested up to 4000 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits
Mapping failed for electromagnetic emission system: litellm.APIError: APIError: OpenAIException - This request req

Processing: 100%|██████████| 400/400 [11:04:18<00:00, 99.65s/it] 

LLM selection failed: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 100 tokens, but can only afford 27. To increase, visit https://openrouter.ai/settings/credits and add more credits', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_33QcGvgPvn2aVps1j9TdBlgP5Ii'}
Progress: 400/400 - Hit rate so far: 39.2%

Pipeline complete! Processed 400 examples.


In [8]:
# Convert to DataFrame and save
results_df = pd.DataFrame(results)

# Save full results
results_df.to_csv('pipeline_results.csv', index=False)
print(f"Results saved to pipeline_results.csv")


Results saved to pipeline_results.csv


## Evaluation Metrics


In [9]:
# Compute metrics for this pipeline
total = len(results_df)
found = (results_df['gold_rank'] > 0).sum()
hit_rate = 100 * found / total

# Mean rank when found
mean_rank = results_df[results_df['gold_rank'] > 0]['gold_rank'].mean()

# MRR (Mean Reciprocal Rank)
mrr = results_df[results_df['gold_rank'] > 0]['gold_rank'].apply(lambda x: 1/x).mean()

print("=" * 60)
print("RAG + LLM SOURCE MAPPING PIPELINE RESULTS")
print("=" * 60)
print(f"\nTotal examples: {total}")
print(f"Gold source found in top 3: {found}/{total} ({hit_rate:.1f}%)")
print(f"Mean gold rank (when found): {mean_rank:.2f}")
print(f"MRR (when found): {mrr:.3f}")
print("=" * 60)


RAG + LLM SOURCE MAPPING PIPELINE RESULTS

Total examples: 400
Gold source found in top 3: 157/400 (39.2%)
Mean gold rank (when found): 1.68
MRR (when found): 0.730


## Compare with Baselines


In [10]:
# Load baseline results for comparison
BASELINES_DIR = "../baselines/"

# Load all baseline CSVs
baseline1_df = pd.read_csv(f"{BASELINES_DIR}baseline1_embedding_top3.csv")
baseline2b_df = pd.read_csv(f"{BASELINES_DIR}baseline2b_name_only.csv")
baseline2c_df = pd.read_csv(f"{BASELINES_DIR}baseline2c_full.csv")
baseline2d_df = pd.read_csv(f"{BASELINES_DIR}baseline2d_name_desc.csv")

print("Baseline results loaded!")


Baseline results loaded!


In [11]:
# Compare all methods
print("=" * 70)
print("COMPARISON: ALL METHODS")
print("=" * 70)

comparison_results = []

# Baseline 1: Embedding Top 3
b1_found = (baseline1_df['gold_rank'] > 0).sum()
b1_total = len(baseline1_df)
b1_mean = baseline1_df[baseline1_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n1. Embedding Top 3:")
print(f"   Gold found: {b1_found}/{b1_total} ({100*b1_found/b1_total:.1f}%)")
print(f"   Mean rank (when found): {b1_mean:.2f}")
comparison_results.append({
    'method': 'Embedding Top 3',
    'found': b1_found, 'total': b1_total,
    'hit_rate': 100*b1_found/b1_total, 'mean_rank': b1_mean
})

# Baseline 2b: LLM + Name Only
b2b_found = (baseline2b_df['gold_rank'] > 0).sum()
b2b_total = len(baseline2b_df)
b2b_mean = baseline2b_df[baseline2b_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n2. LLM Selection (Name Only):")
print(f"   Gold found: {b2b_found}/{b2b_total} ({100*b2b_found/b2b_total:.1f}%)")
print(f"   Mean rank (when found): {b2b_mean:.2f}")
comparison_results.append({
    'method': 'LLM: Name Only',
    'found': b2b_found, 'total': b2b_total,
    'hit_rate': 100*b2b_found/b2b_total, 'mean_rank': b2b_mean
})

# Baseline 2c: LLM + Name + Properties + Description
b2c_found = (baseline2c_df['gold_rank'] > 0).sum()
b2c_total = len(baseline2c_df)
b2c_mean = baseline2c_df[baseline2c_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n3. LLM Selection (Name + Props + Desc):")
print(f"   Gold found: {b2c_found}/{b2c_total} ({100*b2c_found/b2c_total:.1f}%)")
print(f"   Mean rank (when found): {b2c_mean:.2f}")
comparison_results.append({
    'method': 'LLM: Name + Props + Desc',
    'found': b2c_found, 'total': b2c_total,
    'hit_rate': 100*b2c_found/b2c_total, 'mean_rank': b2c_mean
})

# Baseline 2d: LLM + Name + Description
b2d_found = (baseline2d_df['gold_rank'] > 0).sum()
b2d_total = len(baseline2d_df)
b2d_mean = baseline2d_df[baseline2d_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n4. LLM Selection (Name + Desc):")
print(f"   Gold found: {b2d_found}/{b2d_total} ({100*b2d_found/b2d_total:.1f}%)")
print(f"   Mean rank (when found): {b2d_mean:.2f}")
comparison_results.append({
    'method': 'LLM: Name + Desc',
    'found': b2d_found, 'total': b2d_total,
    'hit_rate': 100*b2d_found/b2d_total, 'mean_rank': b2d_mean
})

# Our Pipeline: RAG + LLM Source Mapping
print(f"\n5. RAG + LLM Source Mapping Pipeline (OURS):")
print(f"   Gold found: {found}/{total} ({hit_rate:.1f}%)")
print(f"   Mean rank (when found): {mean_rank:.2f}")
comparison_results.append({
    'method': 'RAG + LLM Source Mapping (OURS)',
    'found': found, 'total': total,
    'hit_rate': hit_rate, 'mean_rank': mean_rank
})

print("\n" + "=" * 70)

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values('hit_rate', ascending=False)
print("\nRanked by Hit Rate:")
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('pipeline_vs_baselines_comparison.csv', index=False)
print("\nComparison saved to pipeline_vs_baselines_comparison.csv")


COMPARISON: ALL METHODS

1. Embedding Top 3:
   Gold found: 160/400 (40.0%)
   Mean rank (when found): 1.75

2. LLM Selection (Name Only):
   Gold found: 129/400 (32.2%)
   Mean rank (when found): 1.65

3. LLM Selection (Name + Props + Desc):
   Gold found: 134/400 (33.5%)
   Mean rank (when found): 1.55

4. LLM Selection (Name + Desc):
   Gold found: 130/400 (32.5%)
   Mean rank (when found): 1.62

5. RAG + LLM Source Mapping Pipeline (OURS):
   Gold found: 157/400 (39.2%)
   Mean rank (when found): 1.68


Ranked by Hit Rate:
                         method  found  total  hit_rate  mean_rank
                Embedding Top 3    160    400     40.00   1.750000
RAG + LLM Source Mapping (OURS)    157    400     39.25   1.675159
       LLM: Name + Props + Desc    134    400     33.50   1.552239
               LLM: Name + Desc    130    400     32.50   1.615385
                 LLM: Name Only    129    400     32.25   1.651163

Comparison saved to pipeline_vs_baselines_comparison.csv


In [12]:
# Show some example results
print("\n=== Sample Results ===")
for i in range(min(5, len(results))):
    r = results[i]
    print(f"\nTarget: {r['target']}")
    print(f"Gold Source: {r['gold_source']}")
    print(f"Selected Sources: {r['selected_sources']}")
    print(f"Gold Rank: {r['gold_rank']}")
    if r['selected_mappings']:
        print(f"Top Source Mappings: {r['selected_mappings'][0]}")



=== Sample Results ===

Target: biological clock
Gold Source: clock
Selected Sources: ['clock', 'electronic scale automatically adjusts', 'Human Body']
Gold Rank: 1
Top Source Mappings: {'changes': 'hands movement', 'state': 'time display', 'adjust': 'time setting'}

Target: Biosphere
Gold Source: Library
Selected Sources: ['Ecosystem', 'forest', 'Desert']
Gold Rank: -1
Top Source Mappings: {'biology': 'organisms', 'biodiversity': 'species', 'ecosystem': 'network'}

Target: Respiratory system
Gold Source: engine
Selected Sources: ['respiration', 'engine', 'air circulation ducts']
Gold Rank: 2
Top Source Mappings: {'oxygen': 'oxygen intake', 'the lungs': 'breathing', 'breathing muscles': 'gas exchange'}

Target: Spread of Pathogens
Gold Source: Spread of Fire
Selected Sources: ['Spread of Fire', 'Immunity', 'crime']
Gold Rank: 1
Top Source Mappings: {'pathogen': 'ignition point', 'crowd': 'flammable materials', 'Prevention and control measures': 'fire fighting methods'}

Target: Gene e